In [1]:
from IPython.display import display, HTML

display(HTML("""
<style>

/* =========================
전체 레이아웃
========================= */

div.container{
    width:85% !important;
}

div.cell.code_cell.rendered{
    width:100%;
}

div.input_prompt{
    padding:0;
}

div.prompt{
    min-width:70px;
}

div#toc-wrapper{
    padding-top:120px;
}

table.dataframe{
    font-size:12px;
}

/* =========================
   코드 입력창
========================= */

div.CodeMirror{
    font-family:"마루 부리OTF 중간" !important;
    font-size:12pt !important;
    line-height:1.6;
}

/* =========================
   입력 셀
========================= */

div.input{
    font-family:"마루 부리OTF 중간" !important;
    font-size:12pt !important;
}

/* =========================
   코드 출력
========================= */

div.output{
    font-family:"마루 부리OTF 중간" !important;
    font-size:12pt !important;
}

/* =========================
   Markdown 전체
========================= */

.rendered_html{
    font-family:"마루 부리OTF 중간" !important;
    font-size:18px !important;
    line-height:1.8;
}

/* 제목 */

.rendered_html h1,
.rendered_html h2,
.rendered_html h3,
.rendered_html h4,
.rendered_html h5,
.rendered_html h6{
    font-family:"마루 부리OTF 조금굵은" !important;
}

/* 본문 */

.rendered_html p{
    font-family:"마루 부리OTF 중간" !important;
}

/* 리스트 */

.rendered_html li{
    font-family:"마루 부리OTF 중간" !important;
    padding:5px;
}

/* 인용 */

.rendered_html blockquote{
    font-family:"마루 부리OTF 중간" !important;
}

/* 표 */

.rendered_html table{
    font-family:"마루 부리OTF 중간" !important;
}

/* 코드 블록 */

.rendered_html pre,
.rendered_html code{
    font-family:"Consolas" !important;
    font-size:12pt !important;
}
table.dataframe{font-size:22px;}
table td, th{font-size:16px;}
table{ margin-left:0 !important;   /* 왼쪽 여백 0 */}
</style>
"""))

# 벡터 DB : Chroma vs. Pinecone
- Chroma : 인메모리 vector DB, 로컬 vector DB, collection단위로 저장
- Pinecone : 클라우드 vector DB, index단위로 저장
(http://www.pinecone.io)에서 api key 받아 .env추가(PINECONE_API_KEY등록)

# 0. 패키지 설치

In [2]:
%pip install -q pinecone langchain-pinecone # --no-warn-script-location

Note: you may need to restart the kernel to use updated packages.


ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^


# 1. knowlege base 구성을 위한 데이터 생성

In [3]:
%%time
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
loader = Docx2txtLoader("tax_docx/소득세법(법률)(제21221호)(20260701).docx")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,   # 문서를 쪼갤 때 1500글자씩 chunking
    chunk_overlap=20, # 20문자씩 overlap 하여 chunking
    # separators=["\n\n", "\n", " ", ""]
)
# Recursive(재귀적으로) 다음의 순서대로 chunking 시도 : "\n\n", "\n", " ", ""
documents = loader.load_and_split(text_splitter=text_splitter)
len(documents)

<timed exec>:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.


CPU times: total: 6.33 s
Wall time: 6.42 s


180

In [4]:
# 임베딩 객체
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
load_dotenv()
embedding = OpenAIEmbeddings(model = 'text-embedding-3-large')

In [5]:
# 임베딩 객체
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings
load_dotenv()
embedding = UpstageEmbeddings(model = 'solar-embedding-1-large-passage')

In [6]:
%%time
# pinecone vector database
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
import os
pc = Pinecone(
    # api_key=os.getenv('PINECONE_API_KEY')
)
# 데이터를 처음 저장할 때
index_name = 'tax-index'
database = PineconeVectorStore.from_documents(
    documents=documents,
    embedding=embedding,
    index_name=index_name
)

ModuleNotFoundError: No module named 'pinecone'

In [7]:
%pip install -q ipywidgets

Note: you may need to restart the kernel to use updated packages.


# 2. 답변 생성을 위한 Retrieval

In [8]:
query = "연봉 5천만원인 직장인은 소득세를 얼마낼까요?"
retrieved_docs = database.similarity_search(query, k=3) # 기본k값은 4

NameError: name 'database' is not defined

In [ ]:
retrieved_doc = "\n--\n".join([doc.page_content.replace('\n\n','\n') for doc in retrieved_docs])
#print(retrieved_doc)

# 3. 답변생성

In [ ]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model='gpt-4.1-nano')

In [ ]:
from langchain_upstage import ChatUpstage
llm = ChatUpstage(model = 'solar-pro2')

In [ ]:
prompt = f"""[identity]
- 당신은 최고의 한국 소득세법 전문가이비다. [context]를 참고해서 사용자 질문에 답하세요. [context]는 다음과 같아요
{retrieved_doc}
- 질문 : {query}"""

In [ ]:
ai_message = llm.invoke(prompt)
print(ai_message.content)